In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import linregress

from armored.models import *
from armored.preprocessing import *

from sklearn.model_selection import KFold, LeaveOneOut

import itertools

from tqdm import tqdm

In [ ]:
# data with initial and end point measurements
df_mono = pd.read_csv("data/exp0/exp0_mono.csv")
df_comm = pd.read_csv("data/exp0/exp0_comm.csv")
df = pd.concat((df_comm, df_mono))
df.head()

In [ ]:
# id species names
species = df.columns.values[2:17]

# consider pH a metabolite
metabolites = df.columns.values[17:18]

# fibers are controls
controls = df.columns.values[18:]

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

In [ ]:
# scale data 
scaler = ZeroMaxScaler(observed, system_variables)
scaler.fit(df)
df_scaled = scaler.transform(df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
data_scaled = format_data(df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), n_metabolites=len(metabolites), n_controls=len(controls), n_hidden=16)
# fit model
brnn.fit(data_scaled, evd_tol=1e-3)

In [ ]:
# full factorial of both species and controls 
n_species = len(species)
n_fibers  = len(controls)

# create matrix of all possible communities
Slist = [np.reshape(np.array(i), (1, n_species)) for i in itertools.product([0, 1], repeat = n_species)]
# remove all zeros community
S = np.array(np.concatenate(Slist)[1:], float)

# create matrix of all possible fiber combinations
Flist = [np.reshape(np.array(i), (1, n_fibers)) for i in itertools.product([0, 1], repeat = n_fibers)]
# remove all zeros combination
F = np.array(np.concatenate(Flist)[1:], float)

# matrix of species and fiber indeces 
M = np.array(np.zeros([S.shape[0]*F.shape[0], 2]), int)
k = 0 
for i in range(S.shape[0]):
    for j in range(F.shape[0]):
        M[k, 0] = int(i)
        M[k, 1] = int(j)
        k += 1

# function to pull sample data 
def gen_exp_cond(k):
    s_ind, f_ind = M[k]
    return S[s_ind], F[f_ind]

# function to generate informative name of experimental condition
def gen_exp_name(Si, Fi):
    exp_name = ""
    for i,si in enumerate(Si):
        if si > 0:
            exp_name += species[i].split("abs")[0] + "-"
    for i,fi in enumerate(Fi):
        if fi > 0:
            exp_name += controls[i] + "-"
    return exp_name[:-1]

In [ ]:
def format_design_data(S, F, t_eval, scaler, batch_size=512):

    # data is a list of tuples (T, X, U, Y, names) where each tuple corresponds to a batch
    data = []
    
    # total number of samples
    n_samples = S.shape[0] * F.shape[0]
    k = 0 
    
    # number of evaluation times
    n_eval = len(t_eval)

    # divide data into batches
    for batch_inds in tqdm(np.array_split(np.arange(n_samples), np.ceil(n_samples / batch_size))):

        # initialize data matrices 
        T = np.empty([len(batch_inds), n_eval])
        T[:] = t_eval
        X = np.empty([len(batch_inds), S.shape[-1]+1])
        X[:] = np.nan
        U = np.empty([len(batch_inds), n_eval, F.shape[-1]])
        U[:] = np.nan

        # keep track of experiment names
        names = []
        for i, batch_ind in enumerate(batch_inds):

            # pull sample data 
            Si, Fi = gen_exp_cond(k)

            # keep track of experiment names
            names.append(gen_exp_name(Si, Fi))

            # store initial condition data
            X[i, :len(Si)] = .01 / sum(Si) * Si
            
            # pH / metabolites
            X[i, len(Si):] = 6.7
            
            # scale system variables 
            X[i] /= scaler.scale_dict_sys[0][:S.shape[-1]+1]
            
            # store controls and observed data
            U[i] =  Fi / sum(Fi)
            
            # scale control variables
            U[i] /= scaler.scale_dict_sys[0][S.shape[-1]+1:]
            
            # update counter
            k += 1

        data.append((T, X, U, names))

    return data

In [ ]:
# format and scale data based on training data
design_data = format_design_data(S, F, np.array([0., 45.]), scaler, batch_size=256)

In [ ]:
# search for best experiments over 3 plates 
designed_experiments = brnn.explore(design_data, n_design=3*96)

In [ ]:
# print set of selected experiments
designed_experiments